# Real-Time Intelligent Surveillance System for Restricted-Zone & Odd-Hour Intrusion Detection

---

## 📋 Problem Statement & Objectives
Modern facility security requires continuous automated monitoring of CCTV surveillance streams to detect unauthorized human presence in restricted zones without generating overwhelming false alarms. 

This project implements an end-to-end, real-time computer vision surveillance architecture that detects, tracks, and evaluates human presence using combined **spatial** and **temporal** evidence:

1. **Human Detection**: Identifies people in high-resolution video streams using **YOLO11**.
2. **Multi-Object Tracking**: Maintains persistent tracking IDs across frames using **ByteTrack**, preventing redundant alert spam.
3. **Spatial Restricted Zone Engine**: Uses a **bottom-center foot contact anchor** $(x_{\text{center}}, y_{\text{max}})$ evaluated with ray-casting point-in-polygon tests (`cv2.pointPolygonTest > 0`), ensuring people leaning near perimeters do not cause false alarms.
4. **Dwell-Time State Machine**: Enforces a continuous presence filter ($\Delta t \ge T_{\text{dwell}}$) before confirming an intrusion, filtering out brief transient edge touches.
5. **Temporal Rule Engine**: Evaluates detections against multi-window weekly schedules, day-of-week policies, and **cross-midnight shifts** (e.g., 22:00 to 06:00).
6. **Rolling Forensic Frame Buffer**: Employs a circular FIFO queue (`collections.deque`) capturing **pre-event** (3 seconds before intrusion) and **post-event** (4 seconds after) video clips asynchronously without reducing inference FPS.
7. **Audit Trail Logging**: Stores structured violation metadata in an SQLite relational database (`surveillance.db`) with links to saved evidence snapshots and MP4 clips.

---

## 🏛️ System Architecture

```
                  [ Video Input Stream ]
     (Webcam / RTSP / MOT17 Surveillance Image Sequence / MP4)
                               │
                               ▼
            [ Frame Preprocessing & Ingestion ]
                               │
                               ▼
        ┌──────────────────────────────────────────────┐
        │        YOLO11 Person Detector (Class 0)      │
        └──────────────────────┬───────────────────────┘
                               │
                               ▼
        ┌──────────────────────────────────────────────┐
        │            ByteTrack Multi-Object            │
        │             Tracking Association             │
        └──────────────────────┬───────────────────────┘
                               │
                ┌──────────────┴──────────────┐
                ▼                             ▼
    [ Foot-Point Extraction ]        [ System Clock / Sim Time ]
    P_foot = ((x1+x2)/2, y2)         Current Timestamp & Day
                │                             │
                ▼                             ▼
    [ Spatial PIP Ray-Casting ]      [ Temporal Schedule Engine ]
    pointPolygonTest(P_foot, Z) > 0  Check Permitted Windows & Shifts
                │                             │
                └──────────────┬──────────────┘
                               │
                               ▼
        ┌──────────────────────────────────────────────┐
        │          Dwell-Time State Machine            │
        │  Δt_dwell = t_current - t_entry >= T_dwell   │
        └──────────────────────┬───────────────────────┘
                               │
                               ▼
        ┌──────────────────────────────────────────────┐
        │         Security Rule Classification         │
        │  • NORMAL             • ENTERING_ZONE        │
        │  • ODD_HOUR_PRESENCE  • ZONE_BREACH          │
        │  • CRITICAL_INTRUSION                        │
        └──────────────────────┬───────────────────────┘
                               │
                ┌──────────────┼──────────────┐
                ▼              ▼              ▼
           [ HUD Bar ]    [ SQLite DB ]  [ Rolling Buffer ]
           Live Visual    Audit Record   Pre/Post Clip &
            Overlays      Persistence    Snapshot Capture
```

---

## 📐 Mathematical Formulation

### 1. Spatial Ground Contact Point
Instead of the bounding box centroid $(\frac{x_1+x_2}{2}, \frac{y_1+y_2}{2})$ (which triggers false alerts when a person bends over a fence), the foot contact point is:
$$P_{\text{foot}} = \left( \frac{x_1 + x_2}{2}, \; y_2 \right)$$

### 2. Ray-Casting Point-in-Polygon (PIP)
Given an arbitrary polygonal restricted zone $\mathcal{P} = \{V_1, V_2, \dots, V_N\}$ with normalized vertices $V_i = (u_i, v_i) \in [0, 1]^2$:
$$\text{Inside}(\mathcal{P}, P_{\text{foot}}) = \begin{cases} 1 & \text{if } \text{cv2.pointPolygonTest}(\mathcal{P}, P_{\text{foot}}, \text{False}) > 0 \\ 0 & \text{otherwise} \end{cases}$$

### 3. Dwell Time State Transition
Let $t_{\text{entry}}(ID)$ be the timestamp when person $ID$ first touches the interior of $\mathcal{P}$:
$$\Delta t_{\text{dwell}}(ID) = t - t_{\text{entry}}(ID)$$
The violation state transition triggers if:
$$\text{State}(ID) = \begin{cases} \text{INTRUSION\_CONFIRMED} & \text{if } \Delta t_{\text{dwell}}(ID) \ge T_{\text{dwell}} \\ \text{ENTERING\_ZONE} & \text{if } 0 < \Delta t_{\text{dwell}}(ID) < T_{\text{dwell}} \\ \text{OUTSIDE} & \text{if } P_{\text{foot}} \notin \mathcal{P} \end{cases}$$

### 4. Cross-Midnight Temporal Intervals
For an operating window $[\tau_{\text{start}}, \tau_{\text{end}}]$ where $\tau_{\text{start}} > \tau_{\text{end}}$ (e.g., $22:00 \to 06:00$):
$$\tau \in \mathcal{W}_{\text{permitted}} \iff \tau \ge \tau_{\text{start}} \;\lor\; \tau \le \tau_{\text{end}}$$

### 5. Event Classification Hierarchy
| Classification | Spatial Location | Time Condition | Dwell Threshold | Action Taken |
| :--- | :--- | :--- | :--- | :--- |
| **`NORMAL`** | Outside Zone | Permitted Hours | N/A | Green bbox; track persistent ID. |
| **`ENTERING_ZONE`** | Inside Zone | Any Hour | $\Delta t < T_{\text{dwell}}$ | Amber bbox; display live dwell countdown. |
| **`ODD_HOUR_PRESENCE`** | Outside Zone | Unauthorized Hours | N/A | Orange bbox; log warning; record snapshot. |
| **`ZONE_BREACH`** | Inside Zone | Permitted Hours | $\Delta t \ge T_{\text{dwell}}$ | Red bbox; log high severity; trigger clip. |
| **`CRITICAL_INTRUSION`** | Inside Zone | Unauthorized Hours | $\Delta t \ge T_{\text{dwell}}$ | Flashing Magenta HUD; log critical alert; record pre/post clip. |


## 1. Environment Setup & Dependency Verification
We verify the execution environment, load dependencies, and verify YOLO11 weights.

In [ ]:
import os
import sys
import time
import json
import yaml
import glob
import zipfile
import sqlite3
import threading
import datetime
from collections import deque
from typing import List, Dict, Any, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from IPython.display import display, HTML, Image

# Set random seed and matplotlib styling
np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 11

device_name = "cuda" if torch.cuda.is_available() else "cpu"
print("=" * 65)
print("SYSTEM ENVIRONMENT CHECK")
print(f"Python Version    : {sys.version.split()[0]}")
print(f"OpenCV Version    : {cv2.__version__}")
print(f"PyTorch Version   : {torch.__version__} (Device: {device_name.upper()})")
print(f"Ultralytics YOLO  : Available")
print("=" * 65)

# Verify local YOLO11 model weights (YOLO11x - Flagship Extra Large Model)
model_path = "yolo11x.pt"
if not os.path.exists(model_path):
    print(f"Downloading {model_path} weights...")
    _ = YOLO("yolo11x.pt")
print(f"YOLO11x weights ready at: {os.path.abspath(model_path)}")


## 2. MOT17 Dataset Ingestion & Ground Truth Parsing
We inspect `MOT17.zip` and `MOT17Labels.zip` located in the project directory, and extract the elevated night-time surveillance sequence **`MOT17-04-SDP`** (1920x1080 @ 30 FPS).

In [ ]:
def inspect_mot17_archives(mot17_zip="MOT17.zip", labels_zip="MOT17Labels.zip"):
    # Inspect MOT17 archives
    print("--- Inspecting MOT17 Archives ---")
    if os.path.exists(labels_zip):
        with zipfile.ZipFile(labels_zip, 'r') as z:
            train_seqs = sorted(list(set(x.split('/')[1] for x in z.namelist() if x.startswith('train/') and '/' in x)))
            print(f"Found {len(train_seqs)} training sequences with Ground Truth in {labels_zip}:")
            print("  ", train_seqs[:8], "...")
    else:
        print(f"Warning: {labels_zip} not found in current directory.")

    if os.path.exists(mot17_zip):
        with zipfile.ZipFile(mot17_zip, 'r') as z:
            names = z.namelist()
            seqs = sorted(list(set(x.split('/')[2] for x in names if len(x.split('/')) > 3 and 'train' in x)))
            print(f"Found {len(seqs)} video sequences in {mot17_zip}")
    else:
        print(f"Warning: {mot17_zip} not found in current directory.")


def extract_mot17_sequence(
    sequence_name="MOT17-04-SDP",
    output_dir="data/mot17_extracted",
    max_frames=300,
    mot17_zip="MOT17.zip",
    labels_zip="MOT17Labels.zip"
) -> Dict[str, Any]:
    # Extracts a subset of frames and ground truth for a specified sequence.
    seq_dir = os.path.join(output_dir, sequence_name)
    img_dir = os.path.join(seq_dir, "img1")
    os.makedirs(img_dir, exist_ok=True)

    extracted_files = []
    
    # 1. Extract frames from MOT17.zip
    if os.path.exists(mot17_zip):
        with zipfile.ZipFile(mot17_zip, 'r') as z:
            prefix = f"MOT17/train/{sequence_name}/img1/"
            frame_entries = sorted([x for x in z.namelist() if x.startswith(prefix) and x.endswith('.jpg')])
            if max_frames:
                frame_entries = frame_entries[:max_frames]
            
            print(f"Extracting {len(frame_entries)} frames for {sequence_name} from {mot17_zip}...")
            for entry in frame_entries:
                fname = os.path.basename(entry)
                target = os.path.join(img_dir, fname)
                if not os.path.exists(target):
                    with open(target, 'wb') as f:
                        f.write(z.read(entry))
                extracted_files.append(target)
    
    # 2. Extract seqinfo.ini and gt.txt from MOT17Labels.zip
    gt_file_path = None
    if os.path.exists(labels_zip):
        with zipfile.ZipFile(labels_zip, 'r') as z:
            gt_entry = f"train/{sequence_name}/gt/gt.txt"
            seq_entry = f"train/{sequence_name}/seqinfo.ini"
            
            if gt_entry in z.namelist():
                gt_file_path = os.path.join(seq_dir, "gt.txt")
                with open(gt_file_path, 'wb') as f:
                    f.write(z.read(gt_entry))
            
            if seq_entry in z.namelist():
                seq_info_path = os.path.join(seq_dir, "seqinfo.ini")
                with open(seq_info_path, 'wb') as f:
                    f.write(z.read(seq_entry))

    print(f"Extracted {len(extracted_files)} frames to {img_dir}")
    print(f"Ground truth path: {gt_file_path}")
    
    return {
        "sequence_name": sequence_name,
        "img_dir": img_dir,
        "frames": sorted(glob.glob(os.path.join(img_dir, "*.jpg"))),
        "gt_file": gt_file_path
    }


def parse_mot17_gt(gt_file_path: str) -> Dict[int, List[Dict[str, Any]]]:
    # Parses MOT17 ground truth format:
    # <frame>, <id>, <bb_left>, <bb_top>, <bb_width>, <bb_height>, <conf>, <class>, <visibility>
    gt_by_frame = {}
    if not gt_file_path or not os.path.exists(gt_file_path):
        return gt_by_frame
    
    with open(gt_file_path, 'r') as f:
        for line in f:
            parts = line.strip().split(',')
            if len(parts) < 8:
                continue
            frame_idx = int(parts[0])
            person_id = int(parts[1])
            x, y, w, h = float(parts[2]), float(parts[3]), float(parts[4]), float(parts[5])
            conf = float(parts[6])
            cls_id = int(parts[7])
            visibility = float(parts[8]) if len(parts) > 8 else 1.0
            
            # Class 1 = Pedestrian
            if cls_id == 1 and conf > 0:
                if frame_idx not in gt_by_frame:
                    gt_by_frame[frame_idx] = []
                gt_by_frame[frame_idx].append({
                    "id": person_id,
                    "bbox": [x, y, x + w, y + h],
                    "foot_point": (x + w / 2.0, y + h),
                    "visibility": visibility
                })
    return gt_by_frame

inspect_mot17_archives()
seq_data = extract_mot17_sequence(sequence_name="MOT17-04-SDP", max_frames=200)
gt_annotations = parse_mot17_gt(seq_data["gt_file"])
print(f"Loaded ground truth for {len(gt_annotations)} frames.")


In [ ]:
# Visualize sample MOT17 frame with Ground Truth annotations
if seq_data["frames"]:
    sample_frame_path = seq_data["frames"][0]
    sample_img = cv2.imread(sample_frame_path)
    sample_rgb = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
    h, w = sample_img.shape[:2]

    # Overlay ground truth bboxes and foot contact points for Frame 1
    frame_gt = gt_annotations.get(1, [])
    vis_img = sample_rgb.copy()
    for gt in frame_gt:
        x1, y1, x2, y2 = [int(v) for v in gt["bbox"]]
        fx, fy = int(gt["foot_point"][0]), int(gt["foot_point"][1])
        cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.circle(vis_img, (fx, fy), 6, (255, 255, 0), -1)
        cv2.putText(vis_img, f"GT ID:{gt['id']}", (x1, max(15, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)

    plt.figure(figsize=(14, 8))
    plt.imshow(vis_img)
    plt.title(f"MOT17-04-SDP Benchmark (Frame 1) - Ground Truth Pedestrian BBoxes & Foot Anchors ({w}x{h})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    print(f"Ground Truth Pedestrians in Frame 1: {len(frame_gt)}")


## 3. System Configuration & Security Rules Setup
Configures detection parameters, temporal schedules (supporting cross-midnight windows), spatial polygons, and rolling buffers.

In [ ]:
SURVEILLANCE_CONFIG = {
    "model": {
        "weights": "yolo11x.pt",
        "confidence": 0.35,
        "iou": 0.45,
        "tracker": "bytetrack.yaml",
        "device": device_name,
        "classes": [0]  # COCO class 0: Person
    },
    "temporal_rules": {
        "mode": "simulated",
        "simulated_start_time": "2026-11-10 22:30:00",  # Nighttime surveillance evaluation
        "schedules": [
            {
                "name": "Weekdays (Mon-Fri)",
                "days": [0, 1, 2, 3, 4],  # 0=Monday, 4=Friday
                "permitted_windows": [
                    {"start": "08:00:00", "end": "18:00:00"}
                ]
            },
            {
                "name": "Saturday (Short Day)",
                "days": [5],
                "permitted_windows": [
                    {"start": "09:00:00", "end": "14:00:00"}
                ]
            },
            {
                "name": "Sunday (Lockdown)",
                "days": [6],
                "permitted_windows": []  # Empty list = full 24-hr lockdown
            }
        ]
    },
    "security_rules": {
        "dwell_time_seconds": 2.0,    # Minimum continuous presence to confirm zone breach
        "cooldown_seconds": 15.0      # Cooldown before logging repeat alerts for the same ID
    },
    "zones": [
        {
            "id": "zone_restricted_plaza",
            "name": "Courtyard Restricted Perimeter",
            "color": [0, 0, 255],     # BGR Red
            "coordinate_type": "normalized",
            "polygon": [
                [0.55, 0.40],
                [0.98, 0.40],
                [0.98, 0.95],
                [0.55, 0.95]
            ]
        }
    ],
    "evidence": {
        "save_snapshot": True,
        "save_video_clip": True,
        "pre_buffer_seconds": 3.0,
        "post_buffer_seconds": 4.0,
        "snapshot_dir": "data/alerts/snapshots",
        "clip_dir": "data/alerts/clips",
        "db_path": "data/surveillance.db"
    },
    "display": {
        "show_hud": True,
        "show_zones": True,
        "show_foot_point": True,
        "show_dwell_timer": True,
        "window_name": "Intelligent Surveillance Stream"
    }
}

os.makedirs(SURVEILLANCE_CONFIG["evidence"]["snapshot_dir"], exist_ok=True)
os.makedirs(SURVEILLANCE_CONFIG["evidence"]["clip_dir"], exist_ok=True)
print("Surveillance configuration initialized.")


## 4. Subsystem 1: Temporal Schedule Engine (`TimeEngine`)
Evaluates real-world or simulated video timestamps against facility operational windows, handling weekly schedules and cross-midnight shifts.

In [ ]:
import datetime
from typing import Dict, Any, Tuple, Optional, List


class TimeEngine:
    """
    Manages temporal security rules, supporting:
    - Multi-window schedules per day of week (0=Mon, 6=Sun)
    - Cross-midnight intervals (e.g., 22:00:00 to 06:00:00)
    - Live system clock or simulated video playback time
    """

    def __init__(self, config: Dict[str, Any], fps: float = 30.0):
        self.mode = config.get("mode", "live")
        self.schedules = config.get("schedules", [])
        self.fps = max(fps, 1.0)
        self.frame_delta = datetime.timedelta(seconds=1.0 / self.fps)

        if self.mode == "simulated":
            start_str = config.get("simulated_start_time", "2026-10-08 22:00:00")
            try:
                self.current_sim_time = datetime.datetime.strptime(
                    start_str, "%Y-%m-%d %H:%M:%S"
                )
            except ValueError:
                self.current_sim_time = datetime.datetime.now()
        else:
            self.current_sim_time = datetime.datetime.now()

    def advance_frame(self) -> datetime.datetime:
        """Advance time for simulated mode by one video frame."""
        if self.mode == "simulated":
            self.current_sim_time += self.frame_delta
            return self.current_sim_time
        return datetime.datetime.now()

    def get_current_time(self) -> datetime.datetime:
        if self.mode == "simulated":
            return self.current_sim_time
        return datetime.datetime.now()

    def _is_time_in_window(
        self, current_t: datetime.time, start_t: datetime.time, end_t: datetime.time
    ) -> bool:
        """
        Check if current_t is within [start_t, end_t].
        Supports overnight / cross-midnight intervals (e.g. 22:00 to 06:00).
        """
        if start_t <= end_t:
            return start_t <= current_t <= end_t
        else:
            # Cross-midnight window (e.g., 22:00:00 to 06:00:00)
            return current_t >= start_t or current_t <= end_t

    def check_temporal_status(
        self, dt: Optional[datetime.datetime] = None
    ) -> Tuple[bool, str]:
        """
        Evaluates whether the timestamp falls outside authorized operating hours.
        Returns:
            (is_odd_hour: bool, description: str)
            True indicates an ODD_HOUR / unauthorized time violation.
        """
        target_dt = dt or self.get_current_time()
        current_day = target_dt.weekday()  # 0=Monday, 6=Sunday
        current_t = target_dt.time()
        day_names = [
            "Monday",
            "Tuesday",
            "Wednesday",
            "Thursday",
            "Friday",
            "Saturday",
            "Sunday",
        ]
        day_name = day_names[current_day]

        # Find matching schedule for current day
        day_schedule = None
        for sched in self.schedules:
            if current_day in sched.get("days", []):
                day_schedule = sched
                break

        # If no schedule rule exists for today, default to strict (unauthorized)
        if not day_schedule:
            return True, f"No authorized schedule configured for {day_name}"

        permitted_windows: List[Dict[str, str]] = day_schedule.get(
            "permitted_windows", []
        )

        # Empty permitted windows means the entire day is locked down / restricted
        if not permitted_windows:
            return True, f"Full 24-hour lockdown active on {day_name}"

        # Check if the time matches ANY permitted window
        for window in permitted_windows:
            try:
                start_parts = [int(p) for p in window["start"].split(":")]
                end_parts = [int(p) for p in window["end"].split(":")]

                start_t = datetime.time(
                    start_parts[0],
                    start_parts[1],
                    start_parts[2] if len(start_parts) > 2 else 0,
                )
                end_t = datetime.time(
                    end_parts[0],
                    end_parts[1],
                    end_parts[2] if len(end_parts) > 2 else 0,
                )

                if self._is_time_in_window(current_t, start_t, end_t):
                    return (
                        False,
                        f"Within authorized window ({window['start']} - {window['end']})",
                    )
            except Exception as e:
                continue

        # If not inside any permitted window, it's an odd-hour detection
        return (
            True,
            f"Outside permitted windows on {day_name} ({target_dt.strftime('%H:%M:%S')})",
        )


# Unit test demonstration for TimeEngine
te = TimeEngine(SURVEILLANCE_CONFIG["temporal_rules"], fps=30.0)
test_thursday_day = datetime.datetime(2026, 11, 12, 14, 0, 0) # Thursday 2 PM
test_thursday_night = datetime.datetime(2026, 11, 12, 23, 0, 0) # Thursday 11 PM
test_sunday = datetime.datetime(2026, 11, 15, 12, 0, 0) # Sunday Lockdown

assert not te.check_temporal_status(test_thursday_day)[0], "Daytime should be authorized"
assert te.check_temporal_status(test_thursday_night)[0], "Nighttime should be odd-hour"
assert te.check_temporal_status(test_sunday)[0], "Sunday should be lockdown"
print("TimeEngine unit tests passed successfully!")


## 5. Subsystem 2: Spatial Restricted Zone Engine (`ZoneEngine`)
Calculates bottom-center foot coordinates and evaluates ray-casting point-in-polygon containment (`cv2.pointPolygonTest > 0`).

In [ ]:
import json
import os
import cv2
import numpy as np
from typing import List, Dict, Any, Optional, Tuple


class RestrictedZone:
    """Represents a single restricted polygonal region."""

    def __init__(self, data: Dict[str, Any]):
        self.id = data.get("id", "zone_unknown")
        self.name = data.get("name", "Restricted Zone")
        self.color = tuple(data.get("color", [0, 0, 255]))  # BGR
        self.coord_type = data.get("coordinate_type", "normalized")
        self.raw_polygon = data.get("polygon", [])
        self._cached_resolution: Optional[Tuple[int, int]] = None
        self._cached_contour: Optional[np.ndarray] = None

    def get_contour(self, frame_width: int, frame_height: int) -> np.ndarray:
        """Returns integer pixel contour scaled to the given frame resolution."""
        if (
            self._cached_contour is not None
            and self._cached_resolution == (frame_width, frame_height)
        ):
            return self._cached_contour

        pts = []
        for pt in self.raw_polygon:
            if self.coord_type == "normalized":
                px = int(pt[0] * frame_width)
                py = int(pt[1] * frame_height)
            else:
                px = int(pt[0])
                py = int(pt[1])
            pts.append([px, py])

        self._cached_resolution = (frame_width, frame_height)
        self._cached_contour = np.array(pts, dtype=np.int32).reshape((-1, 1, 2))
        return self._cached_contour

    def contains_point(
        self, point: Tuple[float, float], frame_width: int, frame_height: int
    ) -> bool:
        """
        Evaluates whether a point (x, y) is strictly inside the polygon.
        pointPolygonTest returns:
          > 0 : strictly inside
          = 0 : on the edge
          < 0 : outside
        """
        contour = self.get_contour(frame_width, frame_height)
        if len(contour) < 3:
            return False
        result = cv2.pointPolygonTest(contour, (float(point[0]), float(point[1])), False)
        return result > 0


class ZoneManager:
    """Manages all configured restricted zones."""

    def __init__(self, zones_path: str):
        self.zones_path = zones_path
        self.zones: List[RestrictedZone] = []
        self.load_zones()

    def load_zones(self):
        self.zones.clear()
        if not os.path.exists(self.zones_path):
            return

        try:
            with open(self.zones_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                for item in data:
                    self.zones.append(RestrictedZone(item))
        except Exception as e:
            print(f"[ZoneManager] Error loading zones from {self.zones_path}: {e}")

    @staticmethod
    def get_foot_point(bbox: List[float]) -> Tuple[float, float]:
        """
        Computes bottom-center foot contact coordinate from [x1, y1, x2, y2].
        """
        x1, y1, x2, y2 = bbox
        foot_x = (x1 + x2) / 2.0
        foot_y = float(y2)
        return (foot_x, foot_y)

    def check_point(
        self, point: Tuple[float, float], frame_width: int, frame_height: int
    ) -> Optional[RestrictedZone]:
        """Returns the first restricted zone containing the foot point, or None."""
        for zone in self.zones:
            if zone.contains_point(point, frame_width, frame_height):
                return zone
        return None

    def draw_zones(self, frame: np.ndarray, alpha: float = 0.25) -> np.ndarray:
        """Renders semi-transparent filled zones with solid colored borders."""
        h, w = frame.shape[:2]
        if not self.zones:
            return frame

        overlay = frame.copy()
        for zone in self.zones:
            contour = zone.get_contour(w, h)
            if len(contour) < 3:
                continue

            # Fill polygon semi-transparently
            cv2.fillPoly(overlay, [contour], zone.color)
            # Outline border
            cv2.polylines(frame, [contour], isClosed=True, color=zone.color, thickness=2)

            # Zone label
            M = cv2.moments(contour)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
            else:
                cx, cy = contour[0][0][0], contour[0][0][1]

            label = f"[ZONE] {zone.name}"
            cv2.putText(
                frame,
                label,
                (cx - 40, cy),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (255, 255, 255),
                2,
                cv2.LINE_AA,
            )

        cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
        return frame


# Preview Zone Overlay on MOT17 Frame
zone_mgr = ZoneManager(SURVEILLANCE_CONFIG["zones_file"] if "zones_file" in SURVEILLANCE_CONFIG else "configs/zones.json")
# Alternatively load directly from config dict
zone_mgr.zones = [RestrictedZone(z) for z in SURVEILLANCE_CONFIG["zones"]]

if seq_data["frames"]:
    test_img = cv2.imread(seq_data["frames"][0])
    test_overlay = zone_mgr.draw_zones(test_img.copy())
    test_rgb = cv2.cvtColor(test_overlay, cv2.COLOR_BGR2RGB)
    
    plt.figure(figsize=(14, 8))
    plt.imshow(test_rgb)
    plt.title("Restricted Zone Overlay on MOT17-04-SDP Surveillance View")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    print("ZoneManager initialized with", len(zone_mgr.zones), "zone(s).")


## 6. Subsystem 3: Rolling Buffer & Asynchronous Clip Recorder (`BufferRecorder`)
Maintains a circular FIFO frame buffer (`collections.deque`) capturing pre-event and post-event video clips asynchronously without blocking inference.

In [ ]:
import os
import cv2
import time
import threading
import numpy as np
from collections import deque
from typing import List, Dict, Any, Optional


class BufferRecorder:
    """
    Maintains a rolling circular buffer of recent frames (pre-event buffer)
    and asynchronously writes pre + post intrusion video clips upon alert triggers.
    """

    def __init__(
        self,
        fps: float,
        frame_size: tuple,
        pre_buffer_seconds: float = 3.0,
        post_buffer_seconds: float = 4.0,
        clip_dir: str = "data/alerts/clips",
        snapshot_dir: str = "data/alerts/snapshots",
    ):
        self.fps = max(fps, 1.0)
        self.frame_size = frame_size  # (width, height)
        self.pre_buffer_len = int(self.fps * pre_buffer_seconds)
        self.post_buffer_len = int(self.fps * post_buffer_seconds)
        self.clip_dir = clip_dir
        self.snapshot_dir = snapshot_dir

        os.makedirs(self.clip_dir, exist_ok=True)
        os.makedirs(self.snapshot_dir, exist_ok=True)

        self.rolling_buffer = deque(maxlen=self.pre_buffer_len)
        self.lock = threading.Lock()

        # Active recording tasks currently collecting post-event frames:
        # List of dict: {
        #   'filename': str,
        #   'frames': list,
        #   'remaining_frames': int,
        #   'fps': float,
        #   'size': tuple
        # }
        self.active_tasks: List[Dict[str, Any]] = []

    def push_frame(self, frame: np.ndarray):
        """Append current frame to circular rolling buffer and active clip tasks."""
        with self.lock:
            # We copy to avoid downstream in-place frame modifications affecting buffer
            frame_copy = frame.copy()
            self.rolling_buffer.append(frame_copy)

            # Feed to active recording sessions
            finished_tasks = []
            for task in self.active_tasks:
                task["frames"].append(frame_copy)
                task["remaining_frames"] -= 1
                if task["remaining_frames"] <= 0:
                    finished_tasks.append(task)

            # Move completed tasks to background writer thread
            for task in finished_tasks:
                self.active_tasks.remove(task)
                threading.Thread(
                    target=self._write_video_worker,
                    args=(task["filename"], task["frames"], task["fps"], task["size"]),
                    daemon=True,
                ).start()

    def trigger_clip(self, clip_filename: str) -> str:
        """
        Initiates an evidence clip capture using the current rolling pre-buffer
        plus upcoming post-buffer frames.
        Returns the output file path.
        """
        out_path = os.path.join(self.clip_dir, clip_filename)
        with self.lock:
            # Snapshot all frames currently in pre-buffer
            pre_frames = list(self.rolling_buffer)

            task = {
                "filename": out_path,
                "frames": pre_frames,
                "remaining_frames": self.post_buffer_len,
                "fps": self.fps,
                "size": self.frame_size,
            }
            self.active_tasks.append(task)

        return out_path

    def save_snapshot(
        self,
        frame: np.ndarray,
        filename: str,
        bbox: Optional[List[float]] = None,
    ) -> str:
        """
        Saves a snapshot of the alert frame. If bbox is provided, also crops
        the target intruder.
        """
        out_path = os.path.join(self.snapshot_dir, filename)
        try:
            # Save full frame
            cv2.imwrite(out_path, frame)
        except Exception as e:
            print(f"[BufferRecorder] Snapshot write failed: {e}")
        return out_path

    @staticmethod
    def _write_video_worker(
        filepath: str, frames: List[np.ndarray], fps: float, size: tuple
    ):
        """Background thread worker to encode video without blocking inference."""
        if not frames:
            return

        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(filepath, fourcc, fps, size)

        if not writer.isOpened():
            # Fallback to XVID AVI if mp4v codec is unavailable on system
            alt_path = filepath.rsplit(".", 1)[0] + ".avi"
            fourcc_avi = cv2.VideoWriter_fourcc(*"XVID")
            writer = cv2.VideoWriter(alt_path, fourcc_avi, fps, size)
            if not writer.isOpened():
                print(f"[BufferRecorder] Failed to open VideoWriter for {filepath}")
                return

        for f in frames:
            if f.shape[1] != size[0] or f.shape[0] != size[1]:
                f = cv2.resize(f, size)
            writer.write(f)

        writer.release()
        print(f"[BufferRecorder] Saved intrusion clip ({len(frames)} frames): {filepath}")


## 7. Subsystem 4: Forensic SQLite Database Logger (`EventDatabase`)
Stores structured audit records in SQLite (`surveillance.db`) with indexes and query helpers.

In [ ]:
import os
import sqlite3
import threading
from typing import Optional, List, Dict, Any


class EventDatabase:
    """
    SQLite database interface for storing and querying intrusion/odd-hour events.
    Thread-safe connection handling.
    """

    def __init__(self, db_path: str = "data/surveillance.db"):
        self.db_path = db_path
        os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
        self.lock = threading.Lock()
        self._init_db()

    def _get_connection(self) -> sqlite3.Connection:
        conn = sqlite3.connect(self.db_path)
        conn.row_factory = sqlite3.Row
        return conn

    def _init_db(self):
        with self.lock:
            conn = self._get_connection()
            try:
                cursor = conn.cursor()
                cursor.execute(
                    """
                    CREATE TABLE IF NOT EXISTS intrusion_events (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        timestamp TEXT NOT NULL,
                        track_id INTEGER NOT NULL,
                        event_type TEXT NOT NULL,
                        severity TEXT NOT NULL,
                        zone_name TEXT,
                        foot_x REAL,
                        foot_y REAL,
                        dwell_time_seconds REAL,
                        snapshot_path TEXT,
                        clip_path TEXT,
                        details TEXT
                    );
                    """
                )
                cursor.execute(
                    "CREATE INDEX IF NOT EXISTS idx_timestamp ON intrusion_events(timestamp);"
                )
                cursor.execute(
                    "CREATE INDEX IF NOT EXISTS idx_event_type ON intrusion_events(event_type);"
                )
                cursor.execute(
                    "CREATE INDEX IF NOT EXISTS idx_track_id ON intrusion_events(track_id);"
                )
                conn.commit()
            finally:
                conn.close()

    def log_event(
        self,
        timestamp: str,
        track_id: int,
        event_type: str,
        severity: str,
        zone_name: Optional[str] = None,
        foot_x: Optional[float] = None,
        foot_y: Optional[float] = None,
        dwell_time_seconds: float = 0.0,
        snapshot_path: Optional[str] = None,
        clip_path: Optional[str] = None,
        details: Optional[str] = None,
    ) -> int:
        """Logs a single security event to the database and returns event ID."""
        with self.lock:
            conn = self._get_connection()
            try:
                cursor = conn.cursor()
                cursor.execute(
                    """
                    INSERT INTO intrusion_events (
                        timestamp, track_id, event_type, severity, zone_name,
                        foot_x, foot_y, dwell_time_seconds, snapshot_path, clip_path, details
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
                    """,
                    (
                        timestamp,
                        track_id,
                        event_type,
                        severity,
                        zone_name or "N/A",
                        round(foot_x, 1) if foot_x is not None else None,
                        round(foot_y, 1) if foot_y is not None else None,
                        round(dwell_time_seconds, 2),
                        snapshot_path,
                        clip_path,
                        details or "",
                    ),
                )
                conn.commit()
                return cursor.lastrowid
            finally:
                conn.close()

    def get_recent_events(self, limit: int = 50) -> List[Dict[str, Any]]:
        """Fetches the most recent security events."""
        with self.lock:
            conn = self._get_connection()
            try:
                cursor = conn.cursor()
                cursor.execute(
                    """
                    SELECT * FROM intrusion_events
                    ORDER BY id DESC LIMIT ?;
                    """,
                    (limit,),
                )
                return [dict(row) for row in cursor.fetchall()]
            finally:
                conn.close()

    def get_dataframe(self, limit: int = 100) -> pd.DataFrame:
        """Returns recent events as a formatted pandas DataFrame."""
        with self.lock:
            conn = self._get_connection()
            try:
                df = pd.read_sql_query(
                    f"SELECT * FROM intrusion_events ORDER BY id DESC LIMIT {limit}", conn
                )
                return df
            finally:
                conn.close()

db_logger = EventDatabase("data/surveillance.db")
print("EventDatabase initialized at:", db_logger.db_path)


## 8. Subsystem 5: Multi-Object Tracking & Dwell State Machine (`SurveillanceTracker`)
Combines YOLO11 person detection, ByteTrack ID association, foot-point PIP spatial testing, dwell-time state transitions, and security classification.

In [ ]:
import time
import datetime
from typing import Dict, Any, List, Optional, Tuple
import numpy as np
from ultralytics import YOLO

# ZoneManager, RestrictedZone defined above
# TimeEngine defined above


class TrackState:
    """Tracks state and dwell time history for a single person ID."""

    def __init__(self, track_id: int, current_time: datetime.datetime):
        self.track_id = track_id
        self.first_seen = current_time
        self.last_seen = current_time

        # Spatial zone tracking
        self.current_zone_id: Optional[str] = None
        self.zone_entry_time: Optional[datetime.datetime] = None
        self.dwell_seconds: float = 0.0

        # Alert tracking
        self.last_alert_time: Optional[datetime.datetime] = None
        self.alerted_events: set = set()

    def update_presence(
        self,
        current_time: datetime.datetime,
        zone: Optional[RestrictedZone],
    ):
        self.last_seen = current_time

        if zone is not None:
            if self.current_zone_id == zone.id:
                # Continuing dwell in same zone
                if self.zone_entry_time is not None:
                    self.dwell_seconds = (
                        current_time - self.zone_entry_time
                    ).total_seconds()
            else:
                # Entered a new zone
                self.current_zone_id = zone.id
                self.zone_entry_time = current_time
                self.dwell_seconds = 0.0
                self.alerted_events.clear()
        else:
            # Person is outside all restricted zones
            self.current_zone_id = None
            self.zone_entry_time = None
            self.dwell_seconds = 0.0


class DetectedPerson:
    """Encapsulates processed information for an individual in a single frame."""

    def __init__(
        self,
        track_id: int,
        bbox: List[float],
        confidence: float,
        foot_point: Tuple[float, float],
        zone: Optional[RestrictedZone],
        dwell_time: float,
        classification: str,
        severity: str,
        is_odd_hour: bool,
    ):
        self.track_id = track_id
        self.bbox = bbox  # [x1, y1, x2, y2]
        self.confidence = confidence
        self.foot_point = foot_point
        self.zone = zone
        self.dwell_time = dwell_time
        self.classification = classification  # NORMAL, ODD_HOUR_PRESENCE, ZONE_BREACH, CRITICAL_INTRUSION
        self.severity = severity  # NORMAL, WARNING, HIGH, CRITICAL
        self.is_odd_hour = is_odd_hour


class SurveillanceTracker:
    """
    Combines YOLO11 detection with ByteTrack tracking,
    dwell-time state machine, and spatial/temporal rule evaluation.
    """

    def __init__(
        self,
        config: Dict[str, Any],
        zone_manager: ZoneManager,
        time_engine: TimeEngine,
    ):
        self.config = config
        model_cfg = config.get("model", {})
        self.weights = model_cfg.get("weights", "yolo11x.pt")
        self.conf = model_cfg.get("confidence", 0.40)
        self.iou = model_cfg.get("iou", 0.45)
        self.tracker_cfg = model_cfg.get("tracker", "bytetrack.yaml")
        self.device = model_cfg.get("device", "cpu")

        rules_cfg = config.get("security_rules", {})
        self.min_dwell_seconds = rules_cfg.get("dwell_time_seconds", 2.0)
        self.cooldown_seconds = rules_cfg.get("cooldown_seconds", 15.0)

        self.zone_manager = zone_manager
        self.time_engine = time_engine

        # Load YOLO model
        print(f"[SurveillanceTracker] Loading model {self.weights} on {self.device}...")
        self.model = YOLO(self.weights)

        # Persistent track states: track_id -> TrackState
        self.track_history: Dict[int, TrackState] = {}

    def process_frame(
        self, frame: np.ndarray, current_time: datetime.datetime
    ) -> Tuple[List[DetectedPerson], List[Dict[str, Any]]]:
        """
        Runs detection, tracking, spatial & temporal classification.
        Returns:
            (persons: List[DetectedPerson], new_alerts: List[Dict])
        """
        h, w = frame.shape[:2]
        is_odd_hour, odd_reason = self.time_engine.check_temporal_status(current_time)

        # Run Ultralytics YOLO with ByteTrack
        results = self.model.track(
            source=frame,
            persist=True,
            tracker=self.tracker_cfg,
            conf=self.conf,
            iou=self.iou,
            classes=[0],  # Person class only
            device=self.device,
            verbose=False,
        )

        detected_persons: List[DetectedPerson] = []
        new_alerts: List[Dict[str, Any]] = []

        if not results or results[0].boxes is None or len(results[0].boxes) == 0:
            return detected_persons, new_alerts

        boxes = results[0].boxes

        for i in range(len(boxes)):
            box = boxes[i]
            # Track ID may be None if tracker is initializing
            if box.id is None:
                continue

            track_id = int(box.id[0].item())
            bbox = box.xyxy[0].cpu().numpy().tolist()  # [x1, y1, x2, y2]
            conf = float(box.conf[0].item())

            foot_pt = self.zone_manager.get_foot_point(bbox)
            zone = self.zone_manager.check_point(foot_pt, frame_width=w, frame_height=h)

            # Update dwell state
            if track_id not in self.track_history:
                self.track_history[track_id] = TrackState(track_id, current_time)

            state = self.track_history[track_id]
            state.update_presence(current_time, zone)

            # Rule Classification
            classification = "NORMAL"
            severity = "NORMAL"
            should_alert = False

            if zone is not None:
                # Inside restricted zone
                if state.dwell_seconds >= self.min_dwell_seconds:
                    if is_odd_hour:
                        classification = "CRITICAL_INTRUSION"
                        severity = "CRITICAL"
                        should_alert = True
                    else:
                        classification = "ZONE_BREACH"
                        severity = "HIGH"
                        should_alert = True
                else:
                    # Dwell threshold not yet reached (approaching / newly entered)
                    classification = "ENTERING_ZONE"
                    severity = "WARNING"
            elif is_odd_hour:
                # Outside restricted zone, but during unauthorized odd hours
                classification = "ODD_HOUR_PRESENCE"
                severity = "WARNING"
                should_alert = True

            # Check cooldown before creating an alert trigger
            if should_alert:
                in_cooldown = False
                if state.last_alert_time is not None:
                    elapsed = (current_time - state.last_alert_time).total_seconds()
                    if elapsed < self.cooldown_seconds:
                        in_cooldown = True

                if not in_cooldown or (
                    classification == "CRITICAL_INTRUSION"
                    and "CRITICAL_INTRUSION" not in state.alerted_events
                ):
                    state.last_alert_time = current_time
                    state.alerted_events.add(classification)

                    alert_info = {
                        "track_id": track_id,
                        "timestamp": current_time.strftime("%Y-%m-%d %H:%M:%S"),
                        "classification": classification,
                        "severity": severity,
                        "zone_name": zone.name if zone else "General Area",
                        "foot_point": foot_pt,
                        "bbox": bbox,
                        "dwell_time": state.dwell_seconds,
                        "odd_hour_reason": odd_reason if is_odd_hour else "",
                        "confidence": conf,
                    }
                    new_alerts.append(alert_info)

            person = DetectedPerson(
                track_id=track_id,
                bbox=bbox,
                confidence=conf,
                foot_point=foot_pt,
                zone=zone,
                dwell_time=state.dwell_seconds,
                classification=classification,
                severity=severity,
                is_odd_hour=is_odd_hour,
            )
            detected_persons.append(person)

        # Cleanup stale tracks (> 60 seconds absent)
        stale_ids = [
            tid
            for tid, tstate in self.track_history.items()
            if (current_time - tstate.last_seen).total_seconds() > 60.0
        ]
        for tid in stale_ids:
            del self.track_history[tid]

        return detected_persons, new_alerts


## 9. Subsystem 6: Professional CCTV Visualizer & HUD (`SurveillanceVisualizer`)
Renders top CCTV telemetry banner, operational status badge, color-coded bounding boxes, foot anchor circles, dwell countdowns, and alert tickers.

In [ ]:
import cv2
import numpy as np
import datetime
from typing import List, Dict, Any, Optional
# DetectedPerson defined above


class SurveillanceVisualizer:
    """
    Renders professional surveillance HUD overlays, status tickers,
    color-coded bounding boxes, dwell timers, and foot contact anchors.
    """

    COLOR_MAP = {
        "NORMAL": (0, 220, 60),               # Green
        "ENTERING_ZONE": (0, 215, 255),        # Amber
        "ODD_HOUR_PRESENCE": (0, 140, 255),    # Orange
        "ZONE_BREACH": (0, 0, 230),            # Red
        "CRITICAL_INTRUSION": (180, 0, 255),   # Magenta/Red
    }

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        disp_cfg = config.get("display", {})
        self.show_hud = disp_cfg.get("show_hud", True)
        self.show_foot = disp_cfg.get("show_foot_point", True)
        self.show_dwell = disp_cfg.get("show_dwell_timer", True)
        self.target_dwell = config.get("security_rules", {}).get(
            "dwell_time_seconds", 2.0
        )

        self.last_alert_msg = ""
        self.last_alert_time: Optional[datetime.datetime] = None

    def set_last_alert(self, msg: str, current_time: datetime.datetime):
        self.last_alert_msg = msg
        self.last_alert_time = current_time

    def render(
        self,
        frame: np.ndarray,
        persons: List[DetectedPerson],
        current_time: datetime.datetime,
        is_odd_hour: bool,
        odd_reason: str,
        total_alerts: int,
    ) -> np.ndarray:
        h, w = frame.shape[:2]

        # 1. Render Person Bounding Boxes & Anchors
        for p in persons:
            color = self.COLOR_MAP.get(p.classification, (0, 255, 0))
            x1, y1, x2, y2 = [int(v) for v in p.bbox]

            # Bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Foot contact point
            if self.show_foot:
                fx, fy = int(p.foot_point[0]), int(p.foot_point[1])
                cv2.circle(frame, (fx, fy), 5, (0, 255, 255), -1)
                cv2.circle(frame, (fx, fy), 8, color, 2)

            # Label banner above bbox
            label = f"ID:{p.track_id} {p.classification}"
            (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
            cv2.rectangle(
                frame,
                (x1, max(0, y1 - lh - 8)),
                (x1 + lw + 6, max(lh + 8, y1)),
                color,
                -1,
            )
            cv2.putText(
                frame,
                label,
                (x1 + 3, max(lh + 4, y1 - 4)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                (0, 0, 0) if p.classification in ["NORMAL", "ENTERING_ZONE"] else (255, 255, 255),
                1,
                cv2.LINE_AA,
            )

            # Dwell time indicator
            if self.show_dwell and p.zone is not None:
                dwell_text = f"Dwell: {p.dwell_time:.1f}s / {self.target_dwell:.1f}s"
                fx, fy = int(p.foot_point[0]), int(p.foot_point[1])
                cv2.putText(
                    frame,
                    dwell_text,
                    (fx - 40, min(h - 10, fy + 18)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45,
                    (0, 255, 255),
                    2,
                    cv2.LINE_AA,
                )

        # 2. Top HUD Bar
        if self.show_hud:
            hud_height = 54
            hud_overlay = frame.copy()
            cv2.rectangle(hud_overlay, (0, 0), (w, hud_height), (20, 20, 20), -1)
            cv2.addWeighted(hud_overlay, 0.82, frame, 0.18, 0, frame)

            # Timestamp
            time_str = current_time.strftime("%Y-%m-%d  %H:%M:%S")
            cv2.putText(
                frame,
                f"SYS TIME: {time_str}",
                (14, 22),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.52,
                (220, 220, 220),
                1,
                cv2.LINE_AA,
            )

            # Operational status indicator
            if is_odd_hour:
                stat_badge = "[ RESTRICTED / ODD-HOURS ]"
                badge_col = (0, 60, 255)  # Bright Red/Orange
            else:
                stat_badge = "[ NORMAL BUSINESS HOURS ]"
                badge_col = (0, 220, 60)  # Green

            cv2.putText(
                frame,
                stat_badge,
                (14, 44),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.52,
                badge_col,
                2,
                cv2.LINE_AA,
            )

            # Metric Counters
            metrics = f"Active People: {len(persons)}  |  Alerts Logged: {total_alerts}"
            (mw, _), _ = cv2.getTextSize(metrics, cv2.FONT_HERSHEY_SIMPLEX, 0.52, 1)
            cv2.putText(
                frame,
                metrics,
                (w - mw - 14, 32),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.52,
                (240, 240, 240),
                1,
                cv2.LINE_AA,
            )

        # 3. Bottom Alert Ticker (active for 3.5s after an alert)
        if self.last_alert_time is not None:
            elapsed = (current_time - self.last_alert_time).total_seconds()
            if 0 <= elapsed <= 3.5:
                ticker_h = 36
                ticker_overlay = frame.copy()
                cv2.rectangle(
                    ticker_overlay,
                    (0, h - ticker_h),
                    (w, h),
                    (0, 0, 180),
                    -1,
                )
                cv2.addWeighted(ticker_overlay, 0.85, frame, 0.15, 0, frame)

                alert_text = f"*** SECURITY ALERT: {self.last_alert_msg} ***"
                cv2.putText(
                    frame,
                    alert_text,
                    (14, h - 12),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA,
                )

        return frame


## 10. Unified Pipeline Execution Engine
Unified execution function `run_surveillance_pipeline()` connecting all engines to any video source, camera, or frame sequence.

In [ ]:
def run_surveillance_pipeline(
    source,
    config: Dict[str, Any] = SURVEILLANCE_CONFIG,
    max_frames: Optional[int] = None,
    save_video_path: Optional[str] = "data/alerts/pipeline_output.mp4"
) -> Dict[str, Any]:
    # Executes the end-to-end intelligent surveillance system.
    # source: int (webcam), str (video file path), or list of str (frame paths).
    is_frame_list = isinstance(source, list)
    
    if is_frame_list:
        if not source:
            raise ValueError("Frame list source is empty.")
        first_frame = cv2.imread(source[0])
        frame_h, frame_w = first_frame.shape[:2]
        fps = 30.0
        total_source_frames = len(source)
    else:
        cap = cv2.VideoCapture(source)
        if not cap.isOpened():
            raise RuntimeError(f"Unable to open video source: {source}")
        fps = cap.get(cv2.CAP_PROP_FPS)
        if fps <= 0 or fps > 120 or np.isnan(fps):
            fps = 30.0
        frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total_source_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print("=" * 65)
    print(" INTELLIGENT SURVEILLANCE SYSTEM ACTIVATING")
    print(f" Resolution: {frame_w}x{frame_h} | FPS: {fps:.1f} | Total Frames: {total_source_frames}")
    print("=" * 65)

    # Initialize subsystems
    zone_mgr = ZoneManager("configs/zones.json" if os.path.exists("configs/zones.json") else None)
    if "zones" in config:
        zone_mgr.zones = [RestrictedZone(z) for z in config["zones"]]

    time_engine = TimeEngine(config.get("temporal_rules", {}), fps=fps)
    db_logger = EventDatabase(config["evidence"]["db_path"])
    
    buffer_rec = BufferRecorder(
        fps=fps,
        frame_size=(frame_w, frame_h),
        pre_buffer_seconds=config["evidence"]["pre_buffer_seconds"],
        post_buffer_seconds=config["evidence"]["post_buffer_seconds"],
        clip_dir=config["evidence"]["clip_dir"],
        snapshot_dir=config["evidence"]["snapshot_dir"]
    )
    
    tracker = SurveillanceTracker(config, zone_mgr, time_engine)
    visualizer = SurveillanceVisualizer(config)

    # Optional video output writer
    out_writer = None
    if save_video_path:
        os.makedirs(os.path.dirname(os.path.abspath(save_video_path)), exist_ok=True)
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        out_writer = cv2.VideoWriter(save_video_path, fourcc, fps, (frame_w, frame_h))

    processed_frames = 0
    total_alerts_count = 0
    start_wall_time = time.time()
    alerts_summary = []

    frame_idx = 0
    while True:
        if is_frame_list:
            if frame_idx >= len(source):
                break
            frame = cv2.imread(source[frame_idx])
            ret = frame is not None
        else:
            ret, frame = cap.read()
        
        if not ret:
            break

        processed_frames += 1
        frame_idx += 1
        current_time = time_engine.advance_frame()

        # Update circular rolling buffer
        buffer_rec.push_frame(frame)

        # Detection, tracking, dwell monitoring, & classification
        persons, new_alerts = tracker.process_frame(frame, current_time)

        # Handle alerts
        for alert in new_alerts:
            total_alerts_count += 1
            tid = alert["track_id"]
            cls_type = alert["classification"]
            zone_name = alert["zone_name"]
            time_str = alert["timestamp"]

            safe_time = time_str.replace(" ", "_").replace(":", "-")
            snap_name = f"alert_{safe_time}_ID{tid}_{cls_type}.jpg"
            clip_name = f"clip_{safe_time}_ID{tid}_{cls_type}.mp4"

            snap_path = None
            clip_path = None

            if config["evidence"]["save_snapshot"]:
                snap_path = buffer_rec.save_snapshot(frame, snap_name)

            if config["evidence"]["save_video_clip"]:
                clip_path = buffer_rec.trigger_clip(clip_name)

            # SQLite log
            db_logger.log_event(
                timestamp=time_str,
                track_id=tid,
                event_type=cls_type,
                severity=alert["severity"],
                zone_name=zone_name,
                foot_x=alert["foot_point"][0],
                foot_y=alert["foot_point"][1],
                dwell_time_seconds=alert["dwell_time"],
                snapshot_path=snap_path,
                clip_path=clip_path,
                details=alert.get("odd_hour_reason", "")
            )

            print(f"[ALERT #{total_alerts_count}] [{cls_type}] Track ID: {tid} | Zone: '{zone_name}' | Dwell: {alert['dwell_time']:.1f}s | {time_str}")
            visualizer.set_last_alert(f"{cls_type} (ID:{tid} in {zone_name})", current_time)
            alerts_summary.append(alert)

        # Draw overlays
        if config["display"]["show_zones"]:
            frame = zone_mgr.draw_zones(frame)

        is_odd_hour, odd_reason = time_engine.check_temporal_status(current_time)
        annotated = visualizer.render(
            frame=frame,
            persons=persons,
            current_time=current_time,
            is_odd_hour=is_odd_hour,
            odd_reason=odd_reason,
            total_alerts=total_alerts_count
        )

        if out_writer:
            out_writer.write(annotated)

        if max_frames and processed_frames >= max_frames:
            break

    if not is_frame_list:
        cap.release()
    if out_writer:
        out_writer.release()

    elapsed = time.time() - start_wall_time
    avg_fps = processed_frames / max(elapsed, 0.001)

    print("\n" + "=" * 65)
    print(" SURVEILLANCE RUN SUMMARY")
    print(f" Total Frames Processed : {processed_frames}")
    print(f" Elapsed Wall Time      : {elapsed:.2f} s")
    print(f" Average Processing FPS : {avg_fps:.2f} FPS")
    print(f" Total Alerts Triggered : {total_alerts_count}")
    print(f" Output Video Path      : {save_video_path}")
    print("=" * 65)

    return {
        "processed_frames": processed_frames,
        "avg_fps": avg_fps,
        "total_alerts": total_alerts_count,
        "alerts_summary": alerts_summary,
        "video_output": save_video_path
    }

print("run_surveillance_pipeline function defined.")


## 11. End-to-End Evaluation on MOT17 Benchmark Dataset
We execute the surveillance system on the extracted `MOT17-04-SDP` sequence.

In [ ]:
# Execute pipeline on the extracted MOT17-04-SDP frames
mot17_results = run_surveillance_pipeline(
    source=seq_data["frames"][:100],
    config=SURVEILLANCE_CONFIG,
    max_frames=100,
    save_video_path="data/alerts/mot17_annotated_evaluation.mp4"
)


## 12. Forensic Evidence Audit: Database Records & Evidence Visualizer
Queries SQLite audit logs, displays the event DataFrame, plots evidence snapshots, and lists saved pre/post video clips.

In [ ]:
# 1. Query SQLite Database and display audit trail
df_events = db_logger.get_dataframe(limit=25)
print(f"Total Logged Events in SQLite: {len(df_events)}")
display(df_events[['id', 'timestamp', 'track_id', 'event_type', 'severity', 'zone_name', 'dwell_time_seconds']])

# 2. Display sample saved snapshots
snapshots = sorted(glob.glob("data/alerts/snapshots/*.jpg"))
print(f"\nTotal Saved Evidence Snapshots: {len(snapshots)}")

if snapshots:
    num_to_show = min(4, len(snapshots))
    fig, axes = plt.subplots(1, num_to_show, figsize=(18, 5))
    if num_to_show == 1:
        axes = [axes]
    for idx, snap_p in enumerate(snapshots[-num_to_show:]):
        img = cv2.imread(snap_p)
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(rgb)
        axes[idx].set_title(f"{os.path.basename(snap_p)[:28]}...", fontsize=9)
        axes[idx].axis("off")
    plt.tight_layout()
    plt.show()

# 3. Check generated video clips
clips = sorted(glob.glob("data/alerts/clips/*.mp4"))
print(f"Total Forensic Pre/Post Video Clips Generated: {len(clips)}")
for c in clips[-5:]:
    size_mb = os.path.getsize(c) / (1024 * 1024)
    print(f"  • {os.path.basename(c)} ({size_mb:.2f} MB)")


## 13. Tracking Accuracy Evaluation Against MOT17 Ground Truth
Calculates Precision, Recall, and F1-Score of YOLO11 pedestrian detections against the official MOT17 annotations at $\text{IoU} \ge 0.50$.

In [ ]:
def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0.0, xB - xA) * max(0.0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    denom = boxAArea + boxBArea - interArea
    return interArea / denom if denom > 0 else 0.0

def evaluate_yolo_on_mot17(frames_list, gt_dict, model, iou_thresh=0.5, max_eval_frames=30):
    total_tp = 0
    total_fp = 0
    total_fn = 0

    eval_frames = frames_list[:max_eval_frames]
    print(f"Evaluating YOLO11 against MOT17 Ground Truth on {len(eval_frames)} frames...")

    for idx, fpath in enumerate(eval_frames):
        frame_idx = idx + 1
        gt_boxes = [g["bbox"] for g in gt_dict.get(frame_idx, [])]
        
        img = cv2.imread(fpath)
        preds = model(img, classes=[0], conf=0.35, verbose=False)
        pred_boxes = preds[0].boxes.xyxy.cpu().numpy().tolist() if preds[0].boxes is not None else []

        matched_gt = set()
        tp = 0
        fp = 0

        for pb in pred_boxes:
            best_iou = 0.0
            best_gt_idx = -1
            for g_idx, gb in enumerate(gt_boxes):
                if g_idx in matched_gt:
                    continue
                iou = compute_iou(pb, gb)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = g_idx

            if best_iou >= iou_thresh and best_gt_idx >= 0:
                tp += 1
                matched_gt.add(best_gt_idx)
            else:
                fp += 1

        fn = len(gt_boxes) - len(matched_gt)
        total_tp += tp
        total_fp += fp
        total_fn += fn

    precision = total_tp / max(total_tp + total_fp, 1)
    recall = total_tp / max(total_tp + total_fn, 1)
    f1 = 2 * (precision * recall) / max(precision + recall, 1e-6)

    print("=" * 50)
    print(" MOT17 BENCHMARK ACCURACY RESULTS")
    print(f" Evaluated Frames : {len(eval_frames)}")
    print(f" True Positives   : {total_tp}")
    print(f" False Positives  : {total_fp}")
    print(f" False Negatives  : {total_fn}")
    print(f" Precision        : {precision:.4f} ({precision*100:.1f}%)")
    print(f" Recall           : {recall:.4f} ({recall*100:.1f}%)")
    print(f" F1-Score         : {f1:.4f} ({f1*100:.1f}%)")
    print("=" * 50)

    # Plot metrics bar chart
    metrics = {"Precision": precision, "Recall": recall, "F1-Score": f1}
    plt.figure(figsize=(8, 4.5))
    bars = plt.bar(metrics.keys(), metrics.values(), color=['#2ca02c', '#1f77b4', '#ff7f0e'], width=0.5)
    plt.ylim(0, 1.1)
    plt.ylabel("Score")
    plt.title(f"YOLO11 Person Detection on MOT17-04-SDP Benchmark (IoU >= {iou_thresh})")
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2.0, yval + 0.02, f"{yval:.3f}", ha='center', va='bottom', fontweight='bold')
    plt.tight_layout()
    plt.show()

# Run evaluation on first 30 frames with YOLO11x
eval_model = YOLO("yolo11x.pt")
evaluate_yolo_on_mot17(seq_data["frames"], gt_annotations, eval_model, iou_thresh=0.5, max_eval_frames=30)


## 14. Interactive Live Webcam / Video File Runner
Utility function to run live surveillance directly on your webcam (`source=0`) or any custom video file.

In [ ]:
def live_camera_surveillance(source=0, max_seconds=10):
    # Runs the surveillance system on a live webcam or video stream for max_seconds.
    print(f"Starting live surveillance on source {source} for {max_seconds} seconds...")
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f"Cannot open camera source: {source}")
        return
    
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    max_frames = int(fps * max_seconds)
    cap.release()

    res = run_surveillance_pipeline(
        source=source,
        config=SURVEILLANCE_CONFIG,
        max_frames=max_frames,
        save_video_path="data/alerts/live_webcam_test.mp4"
    )
    print("Live session completed. Video recorded to:", res["video_output"])

# Uncomment to run on live webcam:
# live_camera_surveillance(source=0, max_seconds=10)
print("Interactive runner ready. Call live_camera_surveillance(source=0) to stream from camera.")


## 15. Interactive Multi-Sequence MOT17 Video Browser & Restricted Zone Editor
This interactive dashboard allows you to:
1. **Select ANY sequence from MOT17** (both train and test sequences in `MOT17.zip`).
2. **Play or scrub through any frame** in the sequence with a live slider.
3. **Interactively edit the restricted region** on-the-fly using polygon presets or custom vertex sliders.
4. **View side-by-side real-time intrusion telemetry**: The left pane renders the CCTV stream with tracking bboxes and zone overlays, while the right pane displays real-time zone occupancy, dwell timers, and chronological breach history.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Scan MOT17.zip for all sequences
def scan_mot17_sequences(mot17_zip_path="MOT17.zip"):
    seq_map = {}
    if os.path.exists(mot17_zip_path):
        with zipfile.ZipFile(mot17_zip_path, "r") as z:
            for name in z.namelist():
                if "/img1/" in name and name.endswith(".jpg"):
                    seq_id = name.split("/img1/")[0].replace("MOT17/", "")
                    if seq_id not in seq_map:
                        seq_map[seq_id] = []
                    seq_map[seq_id].append(name)
        for k in seq_map:
            seq_map[k] = sorted(seq_map[k])
    return seq_map

mot17_all_seqs = scan_mot17_sequences("MOT17.zip")
available_seq_list = sorted(list(mot17_all_seqs.keys())) if mot17_all_seqs else ["train/MOT17-04-SDP"]
default_seq = "train/MOT17-04-SDP" if "train/MOT17-04-SDP" in available_seq_list else available_seq_list[0]

# 2. Interactive Controls
seq_dropdown = widgets.Dropdown(
    options=available_seq_list,
    value=default_seq,
    description="Sequence:",
    layout=widgets.Layout(width="380px")
)

play_widget = widgets.Play(
    value=1,
    min=1,
    max=max(1, len(mot17_all_seqs.get(default_seq, [1]))),
    step=1,
    interval=100,
    description="Play"
)

frame_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=max(1, len(mot17_all_seqs.get(default_seq, [1]))),
    step=1,
    description="Frame:",
    continuous_update=False,
    layout=widgets.Layout(width="450px")
)
widgets.jslink((play_widget, 'value'), (frame_slider, 'value'))

zone_preset = widgets.Dropdown(
    options=["Right Courtyard / Plaza", "Left Walkway", "Central Entry Gate", "Lower Perimeter", "Custom"],
    value="Right Courtyard / Plaza",
    description="Zone Preset:",
    layout=widgets.Layout(width="380px")
)

dwell_slider = widgets.FloatSlider(
    value=2.0, min=0.5, max=5.0, step=0.5,
    description="Dwell (s):",
    layout=widgets.Layout(width="300px")
)

odd_hour_toggle = widgets.Checkbox(
    value=True,
    description="Odd-Hour / Night Lockdown Mode",
    layout=widgets.Layout(width="280px")
)

# Custom Vertex Sliders
v1_x = widgets.FloatSlider(value=0.55, min=0.0, max=1.0, step=0.02, description="TL X:", layout=widgets.Layout(width="200px"))
v1_y = widgets.FloatSlider(value=0.40, min=0.0, max=1.0, step=0.02, description="TL Y:", layout=widgets.Layout(width="200px"))
v2_x = widgets.FloatSlider(value=0.98, min=0.0, max=1.0, step=0.02, description="TR X:", layout=widgets.Layout(width="200px"))
v2_y = widgets.FloatSlider(value=0.40, min=0.0, max=1.0, step=0.02, description="TR Y:", layout=widgets.Layout(width="200px"))
v3_x = widgets.FloatSlider(value=0.98, min=0.0, max=1.0, step=0.02, description="BR X:", layout=widgets.Layout(width="200px"))
v3_y = widgets.FloatSlider(value=0.95, min=0.0, max=1.0, step=0.02, description="BR Y:", layout=widgets.Layout(width="200px"))
v4_x = widgets.FloatSlider(value=0.55, min=0.0, max=1.0, step=0.02, description="BL X:", layout=widgets.Layout(width="200px"))
v4_y = widgets.FloatSlider(value=0.95, min=0.0, max=1.0, step=0.02, description="BL Y:", layout=widgets.Layout(width="200px"))

# Side-by-side Viewport Display
video_display = widgets.Image(format="jpeg", width=720, layout=widgets.Layout(border="2px solid #334155"))
telemetry_display = widgets.HTML(layout=widgets.Layout(width="380px", padding="8px"))

# Session state for widget tracking
widget_track_history = {}
widget_event_log = []
widget_det_cache = {}

def on_seq_change(change):
    new_seq = change['new']
    frames = mot17_all_seqs.get(new_seq, [])
    max_f = max(1, len(frames))
    frame_slider.max = max_f
    play_widget.max = max_f
    frame_slider.value = 1
    widget_track_history.clear()
    widget_event_log.clear()

seq_dropdown.observe(on_seq_change, names='value')

def on_preset_change(change):
    choice = change['new']
    if choice == "Right Courtyard / Plaza":
        v1_x.value, v1_y.value = 0.55, 0.40
        v2_x.value, v2_y.value = 0.98, 0.40
        v3_x.value, v3_y.value = 0.98, 0.95
        v4_x.value, v4_y.value = 0.55, 0.95
    elif choice == "Left Walkway":
        v1_x.value, v1_y.value = 0.02, 0.35
        v2_x.value, v2_y.value = 0.45, 0.35
        v3_x.value, v3_y.value = 0.45, 0.95
        v4_x.value, v4_y.value = 0.02, 0.95
    elif choice == "Central Entry Gate":
        v1_x.value, v1_y.value = 0.35, 0.30
        v2_x.value, v2_y.value = 0.65, 0.30
        v3_x.value, v3_y.value = 0.70, 0.85
        v4_x.value, v4_y.value = 0.30, 0.85
    elif choice == "Lower Perimeter":
        v1_x.value, v1_y.value = 0.10, 0.65
        v2_x.value, v2_y.value = 0.90, 0.65
        v3_x.value, v3_y.value = 0.90, 0.98
        v4_x.value, v4_y.value = 0.10, 0.98

zone_preset.observe(on_preset_change, names='value')

def update_surveillance_view(*args):
    cur_seq = seq_dropdown.value
    cur_frame = frame_slider.value
    frames_list = mot17_all_seqs.get(cur_seq, [])
    if not frames_list or cur_frame > len(frames_list):
        return

    # 1. Read Frame
    frame_path = frames_list[cur_frame - 1]
    with zipfile.ZipFile("MOT17.zip", "r") as z:
        img_bytes = z.read(frame_path)
    frame = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
    h, w = frame.shape[:2]

    # 2. Polygon Definition
    polygon = [
        [v1_x.value, v1_y.value],
        [v2_x.value, v2_y.value],
        [v3_x.value, v3_y.value],
        [v4_x.value, v4_y.value]
    ]

    # 3. Detection & Tracking (Cached for instantaneous polygon reshaping & zero buffering)
    cache_key = (cur_seq, cur_frame)
    if cache_key in widget_det_cache:
        cached_boxes = widget_det_cache[cache_key]
    else:
        preds = eval_model.track(
            source=frame,
            persist=True,
            tracker="bytetrack.yaml",
            conf=0.35,
            iou=0.45,
            classes=[0],
            device=device_name,
            verbose=False
        )
        cached_boxes = []
        if preds and preds[0].boxes is not None and len(preds[0].boxes) > 0:
            for box in preds[0].boxes:
                if box.id is not None:
                    cached_boxes.append((int(box.id[0].item()), box.xyxy[0].cpu().numpy().tolist()))
        widget_det_cache[cache_key] = cached_boxes

    detected_people = []
    zone_intruders = []
    is_odd = odd_hour_toggle.value

    for tid, bbox in cached_boxes:
        foot_pt = ((bbox[0] + bbox[2]) / 2.0, float(bbox[3]))
        
        # PIP check
        poly_scaled = np.array([[int(p[0]*w), int(p[1]*h)] for p in polygon], dtype=np.int32).reshape((-1, 1, 2))
        in_zone = cv2.pointPolygonTest(poly_scaled, foot_pt, False) > 0

        # Dwell state
        key = f"{cur_seq}_{tid}"
        if key not in widget_track_history:
            widget_track_history[key] = {"dwell_frames": 1 if in_zone else 0}
        else:
            if in_zone:
                widget_track_history[key]["dwell_frames"] += 1
            else:
                widget_track_history[key]["dwell_frames"] = 0

        dwell_sec = widget_track_history[key]["dwell_frames"] / 30.0

        if in_zone:
            if dwell_sec >= dwell_slider.value:
                classification = "CRITICAL_INTRUSION" if is_odd else "ZONE_BREACH"
            else:
                classification = "ENTERING_ZONE"
        elif is_odd:
            classification = "ODD_HOUR_PRESENCE"
        else:
            classification = "NORMAL"

        person_data = {
            "id": tid, "bbox": bbox, "foot": foot_pt,
            "in_zone": in_zone, "dwell": dwell_sec, "cls": classification
        }
        detected_people.append(person_data)

        if in_zone:
            zone_intruders.append(person_data)
            if classification in ["ZONE_BREACH", "CRITICAL_INTRUSION"]:
                if not any(e["frame"] == cur_frame and e["id"] == tid for e in widget_event_log):
                    widget_event_log.insert(0, {
                        "frame": cur_frame, "id": tid, "cls": classification, "dwell": f"{dwell_sec:.1f}s"
                    })

    # 4. Render CCTV HUD Frame
    annotated = frame.copy()
    overlay = annotated.copy()
    cv2.fillPoly(overlay, [poly_scaled], (0, 0, 230))
    cv2.polylines(annotated, [poly_scaled], True, (0, 0, 255), 3)
    cv2.addWeighted(overlay, 0.28, annotated, 0.72, 0, annotated)

    COLOR_MAP = {
        "NORMAL": (0, 220, 60), "ENTERING_ZONE": (0, 215, 255),
        "ODD_HOUR_PRESENCE": (0, 140, 255), "ZONE_BREACH": (0, 0, 230), "CRITICAL_INTRUSION": (180, 0, 255)
    }

    for p in detected_people:
        c = COLOR_MAP.get(p["cls"], (0, 255, 0))
        x1, y1, x2, y2 = [int(v) for v in p["bbox"]]
        fx, fy = int(p["foot"][0]), int(p["foot"][1])
        cv2.rectangle(annotated, (x1, y1), (x2, y2), c, 2)
        cv2.circle(annotated, (fx, fy), 5, (0, 255, 255), -1)
        cv2.putText(annotated, f"ID:{p['id']} {p['cls']}", (x1, max(18, y1 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, c, 2)
        if p["in_zone"]:
            cv2.putText(annotated, f"Dwell: {p['dwell']:.1f}s", (fx - 30, min(h - 10, fy + 18)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 255), 2)

    # Top HUD
    cv2.rectangle(annotated, (0, 0), (w, 48), (20, 20, 20), -1)
    status_str = "[ RESTRICTED / ODD-HOURS ]" if is_odd else "[ NORMAL OPERATING HOURS ]"
    status_col = (0, 60, 255) if is_odd else (0, 220, 60)
    cv2.putText(annotated, f"CCTV: {cur_seq} | Frame: {cur_frame}/{len(frames_list)}", (12, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, (220, 220, 220), 1)
    cv2.putText(annotated, status_str, (12, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.50, status_col, 2)
    cv2.putText(annotated, f"People: {len(detected_people)} | Intruders: {len(zone_intruders)}", (w - 240, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.50, (240, 240, 240), 1)

    # Encode to JPEG
    _, encoded = cv2.imencode(".jpg", annotated, [cv2.IMWRITE_JPEG_QUALITY, 80])
    video_display.value = encoded.tobytes()

    # 5. Render Right Telemetry HTML
    intruder_html = ""
    if not zone_intruders:
        intruder_html = "<div style='color: #10B981; font-weight: bold;'>Zone Clear (0 Intruders)</div>"
    else:
        for zi in zone_intruders:
            badge_bg = "#EF4444" if zi["cls"] in ["ZONE_BREACH", "CRITICAL_INTRUSION"] else "#F59E0B"
            intruder_html += (
                f"<div style='background: #FEF2F2; border-left: 4px solid {badge_bg}; padding: 6px 10px; margin-bottom: 6px; border-radius: 4px;'>"
                f"<strong>Person #{zi['id']}</strong>: <span style='color: {badge_bg}; font-weight: bold;'>{zi['cls']}</span><br>"
                f"Dwell: <b>{zi['dwell']:.1f}s</b> / {dwell_slider.value:.1f}s"
                "</div>"
            )

    log_rows = "".join([f"<tr><td>{e['frame']}</td><td><b>#{e['id']}</b></td><td style='color: red;'>{e['cls']}</td><td>{e['dwell']}</td></tr>" for e in widget_event_log[:8]])
    
    telemetry_display.value = (
        "<div style='font-family: sans-serif; background: #F8FAFC; border: 1px solid #E2E8F0; border-radius: 8px; padding: 12px;'>"
        "<h4 style='margin: 0 0 8px 0; color: #1E293B;'>🚨 Real-Time Intrusion Feed</h4>"
        f"<div style='margin-bottom: 12px;'>{intruder_html}</div>"
        "<h5 style='margin: 10px 0 6px 0; color: #334155;'>Chronological Alert Log</h5>"
        "<table style='width: 100%; font-size: 12px; border-collapse: collapse;'>"
        "<thead><tr style='background: #E2E8F0; text-align: left;'><th>Frame</th><th>ID</th><th>Alert</th><th>Dwell</th></tr></thead>"
        f"<tbody>{log_rows if log_rows else '<tr><td colspan=4 style="color: #94A3B8;">No breaches yet</td></tr>'}</tbody>"
        "</table>"
        "</div>"
    )

# Hook callbacks
frame_slider.observe(update_surveillance_view, names='value')
dwell_slider.observe(update_surveillance_view, names='value')
odd_hour_toggle.observe(update_surveillance_view, names='value')
for sld in [v1_x, v1_y, v2_x, v2_y, v3_x, v3_y, v4_x, v4_y]:
    sld.observe(update_surveillance_view, names='value')

# Layout Construction
controls_top = widgets.HBox([seq_dropdown, play_widget, frame_slider])
controls_zone = widgets.HBox([zone_preset, dwell_slider, odd_hour_toggle])
controls_poly = widgets.VBox([
    widgets.HTML("<b>Edit Restricted Zone Vertices (Normalized Coordinates 0.0 - 1.0):</b>"),
    widgets.HBox([v1_x, v1_y, v2_x, v2_y]),
    widgets.HBox([v4_x, v4_y, v3_x, v3_y])
])

side_by_side_viewport = widgets.HBox([video_display, telemetry_display], layout=widgets.Layout(margin="12px 0"))
interactive_dashboard = widgets.VBox([
    widgets.HTML("<h3 style='margin: 0 0 10px 0;'>🛡️ Interactive Surveillance Video Player & Zone Editor</h3>"),
    controls_top,
    controls_zone,
    widgets.Accordion(children=[controls_poly], titles=("📐 Custom Polygon Coordinates Editor",)),
    side_by_side_viewport
])

display(interactive_dashboard)
update_surveillance_view()
